# HW3: Convolutional Neural Networks from Scratch
## Modern Computer Vision, Technion (Spring 2026)

In this assignment, you will implement **Convolutional Neural Networks (CNNs)** from the ground up using only NumPy-style operations and PyTorch's functional unfold/fold utilities.

### Learning Goals
- Understand how convolution works under the hood
- Implement efficient convolution using unfold/fold
- Implement backpropagation through conv and pooling layers
- Build a complete CNN training pipeline
- Train on CIFAR-10 and compare with PyTorch's implementation

### What you've already built (HW2)
- Linear layers (forward + backward)
- ReLU activation
- Cross-entropy loss
- SGD optimizer
- MLP training pipeline

### What you're building today
- **conv2d**: 2D convolution operation (the core operation of CNNs)
- **max_pool2d**: Max pooling for dimension reduction
- **MomentumSGD**: SGD with momentum for better optimization
- **ConvNet**: A complete CNN for CIFAR-10
- **Training pipeline**: Train and evaluate on CIFAR-10

## Setup & Imports

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
from collections import defaultdict

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: CPU (from-scratch implementation)")

[Grand test: skipped execution of this cell for speed]


## Part 0: Provided Code from HW2

These implementations from HW2 are provided so the notebook is self-contained.

### Context Tape Autograd Pattern

We use a context-tape pattern for automatic differentiation:
- Each operation appends `[backward_fn, args]` to a list (ctx)
- `backward()` pops operations in reverse order and applies backward functions
- This implements reverse-mode autodiff (backpropagation)

In [1]:
def ensure_grad(x):
    """Ensure a tensor has requires_grad=True for our autograd."""
    if not x.requires_grad:
        x.requires_grad_(True)
    return x


def backward(x):
    """
    Backward pass through the computation graph.
    Pops operations from ctx in reverse order and applies them.
    """
    if x.grad is None:
        x.grad = torch.ones_like(x)
    
    while hasattr(x, 'ctx') and x.ctx:
        backward_fn, args = x.ctx.pop()
        backward_fn(*args)


class Module:
    """Base class for all neural network modules."""
    
    def __init__(self):
        self.ctx = []  # Computation graph
    
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def parameters(self):
        """Return all parameters of this module and submodules."""
        for attr_name in dir(self):
            attr = getattr(self, attr_name)
            if isinstance(attr, torch.Tensor) and attr.requires_grad:
                yield attr
            elif isinstance(attr, Module):
                yield from attr.parameters()
    
    def train(self):
        """Set module to training mode (not used in from-scratch version)."""
        return self
    
    def eval(self):
        """Set module to evaluation mode."""
        return self
    
    def zero_grad(self):
        """Zero out gradients for all parameters."""
        for param in self.parameters():
            param.grad = None

### ReLU Activation

In [2]:
def relu_forward(x):
    """Forward pass of ReLU: y = max(x, 0)."""
    return torch.clamp(x, min=0)


def relu_backward(dy, x):
    """Backward pass of ReLU: dy/dx = 1 if x > 0 else 0."""
    if x.grad is None:
        x.grad = torch.zeros_like(x)
    x.grad += dy * (x > 0).float()


class ReLU(Module):
    """ReLU activation layer."""
    
    def forward(self, x, ctx=None):
        x = ensure_grad(x)
        y = relu_forward(x)
        y = ensure_grad(y)
        
        if ctx is not None:
            ctx.append([relu_backward, (y.grad, x)])
        
        y.ctx = getattr(x, 'ctx', [])
        return y

### Cross-Entropy Loss

In [3]:
def cross_entropy_forward(logits, targets):
    """
    Forward pass of cross-entropy loss.
    logits: shape (N, C)
    targets: shape (N,) with class indices
    Returns: scalar loss
    """
    N = logits.shape[0]
    # Numerical stability: subtract max
    logits_stable = logits - logits.max(dim=1, keepdim=True)[0]
    log_softmax = logits_stable - torch.log(torch.exp(logits_stable).sum(dim=1, keepdim=True))
    loss = -log_softmax[torch.arange(N), targets].mean()
    return loss


def cross_entropy_backward(logits, targets):
    """
    Backward pass of cross-entropy loss.
    Computes gradient w.r.t. logits.
    """
    N = logits.shape[0]
    logits_stable = logits - logits.max(dim=1, keepdim=True)[0]
    probs = torch.exp(logits_stable) / torch.exp(logits_stable).sum(dim=1, keepdim=True)
    grad = probs.clone()
    grad[torch.arange(N), targets] -= 1
    grad /= N
    if logits.grad is None:
        logits.grad = torch.zeros_like(logits)
    logits.grad += grad


def cross_entropy_loss(logits, targets, ctx=None):
    """
    Cross-entropy loss with autograd support.
    """
    logits = ensure_grad(logits)
    loss = cross_entropy_forward(logits, targets)
    loss = ensure_grad(loss)
    
    if ctx is not None:
        ctx.append([cross_entropy_backward, (logits, targets)])
    
    loss.ctx = getattr(logits, 'ctx', [])
    return loss

### Linear Layer

In [4]:
def linear_forward(x, w, b=None):
    """
    Forward pass of linear layer: y = x @ w^T + b
    x: shape (N, D_in)
    w: shape (D_out, D_in)
    b: shape (D_out,)
    Returns: y of shape (N, D_out)
    """
    y = x @ w.t()
    if b is not None:
        y = y + b
    return y


def linear_backward(dy, x, w, b):
    """
    Backward pass of linear layer.
    dy: shape (N, D_out)
    """
    # Gradient w.r.t. x
    if x.grad is None:
        x.grad = torch.zeros_like(x)
    x.grad += dy @ w
    
    # Gradient w.r.t. w
    if w.grad is None:
        w.grad = torch.zeros_like(w)
    w.grad += dy.t() @ x
    
    # Gradient w.r.t. b
    if b is not None:
        if b.grad is None:
            b.grad = torch.zeros_like(b)
        b.grad += dy.sum(dim=0)


class Linear(Module):
    """Linear (fully connected) layer."""
    
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Initialize weights with He initialization
        w = torch.randn(out_features, in_features) * np.sqrt(2.0 / in_features)
        self.w = ensure_grad(w)
        
        if bias:
            self.b = ensure_grad(torch.zeros(out_features))
        else:
            self.b = None
    
    def forward(self, x, ctx=None):
        x = ensure_grad(x)
        y = linear_forward(x, self.w, self.b)
        y = ensure_grad(y)
        
        if ctx is not None:
            ctx.append([linear_backward, (y.grad, x, self.w, self.b)])
        
        y.ctx = getattr(x, 'ctx', [])
        return y

### SGD Optimizer

In [5]:
class SGD:
    """Stochastic Gradient Descent optimizer."""
    
    def __init__(self, params, lr=0.01):
        self.params = list(params)
        self.lr = lr
    
    def step(self):
        """Update parameters using their gradients."""
        for param in self.params:
            if param.grad is not None:
                param.data -= self.lr * param.grad
    
    def zero_grad(self):
        """Zero out gradients."""
        for param in self.params:
            param.grad = None

### Training & Evaluation Utilities

In [6]:
def accuracy(logits, targets):
    """Compute classification accuracy.
    logits: shape (N, C)
    targets: shape (N,)
    """
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def evaluate(model, dataloader, loss_fn):
    """Evaluate model on a dataset.
    Returns: (loss, accuracy)
    """
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    model.eval()
    with torch.no_grad():
        for x, y in dataloader:
            logits = model.forward(x)
            loss = loss_fn(logits, y)
            
            total_loss += loss.item() * x.shape[0]
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_samples += x.shape[0]
    
    return total_loss / total_samples, total_correct / total_samples


def train_epoch(model, train_loader, optimizer, loss_fn):
    """Train for one epoch.
    Returns: (average_loss, accuracy)
    """
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    for x, y in train_loader:
        # Forward pass
        logits = model.forward(x)
        loss = loss_fn(logits, y)
        
        # Backward pass
        model.zero_grad()
        backward(loss)
        
        # Update
        optimizer.step()
        
        # Statistics
        total_loss += loss.item() * x.shape[0]
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += x.shape[0]
    
    return total_loss / total_samples, total_correct / total_samples

---
# Part 1: Convolution (conv2d)

## Overview

Convolution is the core operation of CNNs. A convolution applies a small filter (kernel) across an input image, sliding it over every position and computing element-wise products.

### High-level idea

For an image `x` of shape `(N, C_in, H, W)` and kernel `w` of shape `(C_out, C_in, kH, kW)`:

1. **Extract patches**: Use `F.unfold` to extract all kernel-sized patches from the input
   - Output shape: `(N, C_in * kH * kW, L)` where L is the number of patches

2. **Reshape kernel**: Flatten the kernel from `(C_out, C_in, kH, kW)` to `(C_out, C_in * kH * kW)`

3. **Matrix multiply**: Each patch is a column in the unfolded input. Multiply kernel with these patches:
   - `y_flat = w_flat @ x_unfold` → shape `(N, C_out, L)`

4. **Reshape output**: Reshape `y_flat` back to `(N, C_out, H_out, W_out)`

5. **Add bias**: Add bias term to the output

### Output spatial dimensions

```
H_out = floor((H + 2*padding - dilation*(kH-1) - 1) / stride + 1)
W_out = floor((W + 2*padding - dilation*(kW-1) - 1) / stride + 1)
```

### Why unfold/fold?

- Converts convolution to a series of matrix multiplications
- Much faster than explicit loops
- Makes backpropagation straightforward: just compute gradients of matrix ops
- The gradients automatically scatter back to input positions via fold

## Forward Pass: Implementing conv2d_forward

Your task: Implement the forward pass of 2D convolution using `F.unfold` and `F.fold`.

**Steps:**
1. Use `F.unfold` to extract patches from the input
2. Reshape the kernel to 2D for matrix multiplication
3. Compute the matrix product
4. Reshape the output to the correct spatial dimensions
5. Add the bias if provided

**Hints:**
- `F.unfold` returns shape `(N, C*kH*kW, L)` where L = H_out * W_out
- You can compute H_out and W_out using the formula above
- `F.fold` takes `(N, C, L)` tensor and kernel_size and unfolds it back to `(N, C, H, W)`
- For bias, use broadcasting (add to shape `(N, C_out, 1, 1)` to broadcast across spatial dims)

In [7]:
def conv2d_forward(x, w, b=None, padding=0, stride=1, dilation=1):
    """
    Forward pass of 2D convolution using unfold.

    Args:
        x: input tensor of shape (B, C_in, H, W)
        w: weight tensor of shape (C_out, C_in, K_h, K_w)
        b: optional bias of shape (C_out,)
        padding: padding size (same for all sides)
        stride: stride size (same for H and W)
        dilation: dilation factor for the kernel

    Returns:
        output tensor of shape (B, C_out, H_out, W_out)
    """
    B, C_in, H, W = x.shape
    C_out, _, K_h, K_w = w.shape

    # Unfold the input
    x_unfolded = F.unfold(
        x,
        kernel_size=(K_h, K_w),
        padding=padding,
        stride=stride,
        dilation=dilation
    )  # Shape: (B, C_in*K_h*K_w, L)

    # Reshape weights for matrix multiplication
    w_reshaped = w.view(C_out, -1)  # Shape: (C_out, C_in*K_h*K_w)

    # Perform convolution as matrix multiplication
    # (C_out, C_in*K_h*K_w) @ (B, C_in*K_h*K_w, L) = (B, C_out, L)
    output = w_reshaped @ x_unfolded  # Shape: (B, C_out, L)

    # Add bias if provided
    if b is not None:
        output = output + b.view(1, -1, 1)

    # Compute output spatial dimensions
    H_out = (H + 2 * padding - dilation * (K_h - 1) - 1) // stride + 1
    W_out = (W + 2 * padding - dilation * (K_w - 1) - 1) // stride + 1

    # Reshape to (B, C_out, H_out, W_out)
    output = output.view(B, C_out, H_out, W_out)

    return output

## Sanity Check: Forward Pass

Compare your implementation with PyTorch's `F.conv2d`.

In [ ]:
# Test conv2d forward
torch.manual_seed(42)
N, C_in, H, W = 2, 3, 8, 8
C_out, kH, kW = 4, 3, 3
padding, stride = 1, 1

x_test = torch.randn(N, C_in, H, W)
w_test = torch.randn(C_out, C_in, kH, kW)
b_test = torch.randn(C_out)

# Your implementation
# y_yours = conv2d_forward(x_test, w_test, b_test, padding=padding, stride=stride)

# PyTorch reference
y_torch = F.conv2d(x_test, w_test, b_test, padding=padding, stride=stride)

# print(f"Your output shape: {y_yours.shape}")
# print(f"PyTorch output shape: {y_torch.shape}")
# print(f"Max difference: {(y_yours - y_torch).abs().max().item()}")
# assert torch.allclose(y_yours, y_torch, atol=1e-5), "Forward pass doesn't match PyTorch!"
# print("✓ Forward pass matches PyTorch!")

[Grand test: skipped execution of this cell for speed]


## Backward Pass: Implementing conv2d_backward

The backward pass computes gradients w.r.t. input, weights, and bias.

**Key insight:** If forward is `y = (x_unfold @ w_flat.T) + b`, then:

- **Gradient w.r.t. w**: `w.grad = y.grad @ x_unfold.T` (after unfolding y.grad)
- **Gradient w.r.t. x**: Use `F.fold` on `(w.T @ y_grad_flat)` to scatter gradients back to input positions
- **Gradient w.r.t. b**: Sum y.grad across batch and spatial dimensions

Your task: Implement the backward pass.

**Hints:**
- You'll need to call `F.unfold` again on the input (you have it in the context)
- Use `F.fold` with `kernel_size=(1, 1)` to "unpad" the gradients back to input size, or use padding/stride to account for the original convolution params
- Be careful about which tensors get the gradients (w.grad, x.grad, b.grad)

In [8]:
def conv2d_backward(dy, x, w, b, padding, stride, dilation):
    """
    Backward pass for 2D convolution.

    Args:
        dy: gradient w.r.t. output of shape (B, C_out, H_out, W_out)
        x: input tensor of shape (B, C_in, H, W)
        w: weight tensor of shape (C_out, C_in, K_h, K_w)
        b: bias tensor of shape (C_out,) or None
        padding, stride, dilation: same as forward pass

    Returns:
        dx, dw, db: gradients w.r.t. x, w, b
    """
    B, C_in, H, W = x.shape
    C_out, _, K_h, K_w = w.shape
    B, C_out, H_out, W_out = dy.shape

    # Flatten dy for computation
    dy_flat = dy.view(B, C_out, -1)  # (B, C_out, H_out*W_out)

    # Unfold x
    x_unfolded = F.unfold(
        x,
        kernel_size=(K_h, K_w),
        padding=padding,
        stride=stride,
        dilation=dilation
    )  # (B, C_in*K_h*K_w, H_out*W_out)

    # Compute dw: (C_out, C_in*K_h*K_w) from (B, C_out, H_out*W_out) @ (B, C_in*K_h*K_w, H_out*W_out).T
    w_reshaped = w.view(C_out, -1)
    dw = (dy_flat @ x_unfolded.transpose(1, 2)).sum(0)  # Sum over batch
    dw = dw.view(w.shape)

    # Compute db: sum of dy over spatial dimensions
    if b is not None:
        db = dy_flat.sum(dim=(0, 2))  # Sum over batch and spatial
    else:
        db = None

    # Compute dx: use col2im/fold
    w_t = w.view(C_out, -1).t()  # (C_in*K_h*K_w, C_out)
    dcol = w_t @ dy_flat  # (B, C_in*K_h*K_w, H_out*W_out)

    dx = F.fold(
        dcol,
        output_size=(H, W),
        kernel_size=(K_h, K_w),
        padding=padding,
        stride=stride,
        dilation=dilation
    )  # (B, C_in, H, W)

    return dx, dw, db

## Sanity Check: Backward Pass

Compare your gradients with PyTorch's autograd.

In [ ]:
# Test conv2d backward
torch.manual_seed(42)
N, C_in, H, W = 2, 3, 8, 8
C_out, kH, kW = 4, 3, 3
padding, stride = 1, 1

x_test = torch.randn(N, C_in, H, W, requires_grad=True)
w_test = torch.randn(C_out, C_in, kH, kW, requires_grad=True)
b_test = torch.randn(C_out, requires_grad=True)

# PyTorch reference
y_torch = F.conv2d(x_test, w_test, b_test, padding=padding, stride=stride)
loss_torch = y_torch.sum()  # dummy loss
loss_torch.backward()

# Your gradients should match:
# x.grad, w.grad, b.grad from your conv2d_backward
# print(f"PyTorch x.grad shape: {x_test.grad.shape}")
# print(f"PyTorch w.grad shape: {w_test.grad.shape}")
# print(f"PyTorch b.grad shape: {b_test.grad.shape}")
# print("Compare your gradients with these PyTorch gradients")

[Grand test: skipped execution of this cell for speed]


## Conv2d Layer Wrapper

Wrap the conv2d forward and backward functions in a Module class.

### Your Task: Implement Conv2d Layer

Create a `Conv2d` module that wraps your `conv2d_forward` and `conv2d_backward` functions.

**Requirements:**
- Constructor: Initialize weights and bias
- Weight initialization: Use He initialization (scale by sqrt(2 / (C_in * kH * kW)))
- `forward` method: Call your conv2d function and store context for backward
- Return output tensor with requires_grad=True

In [9]:
class Conv2d(Module):
    """2D Convolution layer."""
    
    def __init__(self, in_channels, out_channels, kernel_size, padding=0, stride=1, dilation=1, bias=True):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        k = 1.0 / (in_channels * kernel_size[0] * kernel_size[1])
        self.weight = (torch.rand(out_channels, in_channels, kernel_size[0], kernel_size[1]) * 2 - 1) * (k ** 0.5)
        self.weight.requires_grad_(True)
        self.bias = None
        if bias:
            self.bias = (torch.rand(out_channels) * 2 - 1) * (k ** 0.5)
            self.bias.requires_grad_(True)
        self.padding = padding
        self.stride = stride
        self.dilation = dilation
    
    def forward(self, x, ctx=None):
        y = conv2d_forward(x, self.weight, self.bias, self.padding, self.stride, self.dilation)
        if ctx is not None:
            ctx.append([conv2d_backward, [y, x, self.weight, self.bias, self.padding, self.stride, self.dilation]])
        return y
    
    def parameters(self):
        params = [self.weight]
        if self.bias is not None:
            params.append(self.bias)
        return params

---
# Part 2: Max Pooling (max_pool2d)

## Overview

Max pooling reduces spatial dimensions by taking the maximum value in each non-overlapping (or overlapping with stride) kernel window.

### Key idea

1. Use `F.unfold` to extract patches (same as conv2d)
2. Take max along the channel dimension (the kernel values)
3. Also keep track of argmax indices (needed for backward)
4. Reshape back to spatial dimensions

### Backward

- Gradient only flows to the position that was the max
- Use the argmax indices to scatter gradients back
- Use `F.fold` to accumulate gradients to the input positions

In [10]:
def max_pool2d_forward(x, kernel_size, padding=0, stride=None):
    """
    Forward pass of 2D max pooling.
    Returns output and indices for backward pass.
    """
    if stride is None:
        stride = kernel_size
    output, indices = F.max_pool2d(x, kernel_size=kernel_size, padding=padding, 
                                    stride=stride, return_indices=True)
    return output, indices

In [11]:
def max_pool2d_backward(dy, x, kernel_size, padding=0, stride=None, argmax_indices=None):
    """
    Backward pass for 2D max pooling.
    Uses F.max_unpool2d with indices from the forward pass.
    """
    if stride is None:
        stride = kernel_size
    
    # Recompute forward to get indices
    _, indices = F.max_pool2d(x, kernel_size=kernel_size, padding=padding, 
                              stride=stride, return_indices=True)
    
    # Use max_unpool2d to scatter gradients back
    dx = F.max_unpool2d(dy, indices, kernel_size=kernel_size, padding=padding, 
                        stride=stride, output_size=x.shape)
    return dx

### MaxPool2d Layer

In [12]:
class MaxPool2d(Module):
    """2D Max Pooling layer."""
    
    def __init__(self, kernel_size, padding=0, stride=None):
        super().__init__()
        self.kernel_size = kernel_size
        self.padding = padding
        self.stride = stride if stride is not None else kernel_size
    
    def forward(self, x, ctx=None):
        output, indices = max_pool2d_forward(x, self.kernel_size, self.padding, self.stride)
        if ctx is not None:
            ctx.append([max_pool2d_backward, [output, x, self.kernel_size, self.padding, self.stride, indices]])
        return output
    
    def parameters(self):
        return []

## Sanity Check: Max Pooling

In [13]:
# Test max_pool2d forward
torch.manual_seed(42)
x_test = torch.randn(2, 3, 8, 8)

# Your implementation
# y_yours = max_pool2d_forward(x_test, kernel_size=2, stride=2)

# PyTorch reference
y_torch = F.max_pool2d(x_test, kernel_size=2, stride=2)

# print(f"Your output shape: {y_yours.shape}")
# print(f"PyTorch output shape: {y_torch.shape}")
# print(f"Max difference: {(y_yours - y_torch).abs().max().item()}")
# assert torch.allclose(y_yours, y_torch, atol=1e-5), "Max pooling doesn't match PyTorch!"
# print("✓ Max pooling matches PyTorch!")

NameError: name 'torch' is not defined

---
# Part 3: MomentumSGD Optimizer

## Overview

SGD with momentum accumulates gradients to smooth the optimization trajectory:

```
v_t = momentum * v_{t-1} + grad
param -= lr * v_t
```

This helps:
- Accelerate convergence in consistent directions
- Reduce oscillations in noisy gradients
- Escape shallow local minima

In [14]:
class MomentumSGD:
    """
    SGD with momentum optimizer.
    """
    def __init__(self, params, lr=0.01, momentum=0.9):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.velocity = [torch.zeros_like(p) for p in self.params]

    def step(self):
        for param, vel in zip(self.params, self.velocity):
            if param.grad is not None:
                vel.mul_(self.momentum).add_(param.grad, alpha=self.lr)
                param.data.sub_(vel)

    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()

---
# Part 4: ConvNet Model for CIFAR-10

## Architecture Design

Design a CNN for CIFAR-10 classification. CIFAR-10 has:
- 32x32 RGB images (3 channels)
- 10 output classes

### Suggested Architecture

```
Conv2d(3, 32, kernel_size=3, padding=1)
  → ReLU
  → MaxPool2d(2, stride=2)  # 32x32 → 16x16
  
Conv2d(32, 64, kernel_size=3, padding=1)
  → ReLU
  → MaxPool2d(2, stride=2)  # 16x16 → 8x8
  
Conv2d(64, 128, kernel_size=3, padding=1)
  → ReLU
  → MaxPool2d(2, stride=2)  # 8x8 → 4x4
  
Flatten → Linear(128*4*4, 256) → ReLU → Linear(256, 10)
```

Feel free to adjust the architecture (more/fewer channels, different kernel sizes, etc.)

In [15]:
class ConvNet(Module):
    """
    Simple convolutional network for CIFAR-10.
    """
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = Linear(64 * 8 * 8, 128)
        self.fc2 = Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = relu_forward(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = relu_forward(x)
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = relu_forward(x)
        x = self.fc2(x)
        return x

---
# Part 5: Load CIFAR-10 Dataset

In [ ]:
# Download and prepare CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

print("Loading CIFAR-10 dataset...")
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Use smaller batch size for from-scratch CPU implementation
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")

[Grand test: skipped execution of this cell for speed]


---
# Part 6: Training Loop

Train your ConvNet on CIFAR-10 for a few epochs.

In [ ]:
# Initialize model and optimizer
model = ConvNet()
optimizer = MomentumSGD(model.parameters(), lr=0.01, momentum=0.9)
loss_fn = cross_entropy_loss

# Training configuration
num_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Starting training...")
for epoch in range(num_epochs):
    start_time = time.time()
    
    # Train one epoch
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    # Evaluate on test set
    test_loss, test_acc = evaluate(model, test_loader, loss_fn)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{num_epochs} ({elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

print(f"\nTraining complete! Final test accuracy: {history['test_acc'][-1]:.4f}")

[Grand test: skipped execution of this cell for speed]


## Visualize Training Progress

In [ ]:
# Plot loss and accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['test_loss'], label='Test Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy curve
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['test_acc'], label='Test Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Classification Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nBest test accuracy: {max(history['test_acc']):.4f}")

[Grand test: skipped execution of this cell for speed]


## Sample Predictions

Visualize some predictions from your model.

In [ ]:
# Get some test samples
model.eval()
cifar10_classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Get a batch
x_batch, y_batch = next(iter(test_loader))
with torch.no_grad():
    logits = model.forward(x_batch[:9])
    preds = logits.argmax(dim=1)

# Visualize
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    # Unnormalize image
    img = x_batch[i].numpy().transpose(1, 2, 0)
    img = (img + 1) / 2  # Undo normalization
    img = np.clip(img, 0, 1)
    
    ax.imshow(img)
    true_label = cifar10_classes[y_batch[i]]
    pred_label = cifar10_classes[preds[i]]
    color = 'green' if preds[i] == y_batch[i] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

[Grand test: skipped execution of this cell for speed]


---
# Part 7: Comparison with PyTorch

Implement the same ConvNet architecture using PyTorch's built-in modules and compare performance.

In [16]:
class ConvNet(Module):
    """
    Simple convolutional network for CIFAR-10.
    """
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = Linear(64 * 8 * 8, 128)
        self.fc2 = Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = relu_forward(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = relu_forward(x)
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = relu_forward(x)
        x = self.fc2(x)
        return x

In [ ]:
# Train PyTorch version
torch.manual_seed(42)
model_pt = ConvNetPyTorch()
optimizer_pt = torch.optim.SGD(model_pt.parameters(), lr=0.01, momentum=0.9)
loss_fn_pt = nn.CrossEntropyLoss()

num_epochs_pt = 5
history_pt = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Training PyTorch version...")
for epoch in range(num_epochs_pt):
    # Train
    model_pt.train()
    train_loss, train_correct = 0.0, 0
    for x, y in train_loader:
        logits = model_pt(x)
        loss = loss_fn_pt(logits, y)
        
        optimizer_pt.zero_grad()
        loss.backward()
        optimizer_pt.step()
        
        train_loss += loss.item() * x.shape[0]
        train_correct += (logits.argmax(dim=1) == y).sum().item()
    
    train_loss /= len(train_dataset)
    train_acc = train_correct / len(train_dataset)
    
    # Evaluate
    model_pt.eval()
    test_loss, test_correct = 0.0, 0
    with torch.no_grad():
        for x, y in test_loader:
            logits = model_pt(x)
            loss = loss_fn_pt(logits, y)
            test_loss += loss.item() * x.shape[0]
            test_correct += (logits.argmax(dim=1) == y).sum().item()
    
    test_loss /= len(test_dataset)
    test_acc = test_correct / len(test_dataset)
    
    history_pt['train_loss'].append(train_loss)
    history_pt['train_acc'].append(train_acc)
    history_pt['test_loss'].append(test_loss)
    history_pt['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs_pt} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

print(f"\nPyTorch training complete! Final test accuracy: {history_pt['test_acc'][-1]:.4f}")

[Grand test: skipped execution of this cell for speed]


## Comparison Plot

In [ ]:
# Compare accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# From-scratch vs PyTorch test accuracy
axes[0].plot(history['test_acc'], label='From-scratch', marker='o', linewidth=2)
axes[0].plot(history_pt['test_acc'], label='PyTorch', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy Comparison')
axes[0].legend()
axes[0].grid(True)

# From-scratch vs PyTorch train loss
axes[1].plot(history['train_loss'], label='From-scratch', marker='o', linewidth=2)
axes[1].plot(history_pt['train_loss'], label='PyTorch', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Train Loss')
axes[1].set_title('Train Loss Comparison')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"From-scratch test accuracy: {history['test_acc'][-1]:.4f}")
print(f"PyTorch test accuracy:      {history_pt['test_acc'][-1]:.4f}")
print(f"\nDifference: {abs(history['test_acc'][-1] - history_pt['test_acc'][-1]):.4f}")

[Grand test: skipped execution of this cell for speed]


---
# Summary

## What you implemented

1. **conv2d**: The core convolution operation using unfold/fold for efficiency
2. **conv2d_backward**: Efficient backpropagation through convolution
3. **Conv2d Layer**: A Module wrapper around the conv2d function
4. **max_pool2d**: Max pooling for spatial dimension reduction
5. **max_pool2d_backward**: Backpropagation through max pooling
6. **MaxPool2d Layer**: A Module wrapper
7. **MomentumSGD**: SGD optimizer with momentum
8. **ConvNet**: A complete CNN architecture for CIFAR-10
9. **Training Pipeline**: Training and evaluation on CIFAR-10
10. **Comparison**: Validated your implementation against PyTorch

## Key insights

- **Unfold/Fold pattern**: Converts convolution into matrix operations for efficiency
- **Context-tape autograd**: Simple but effective automatic differentiation
- **Momentum optimization**: Speeds up convergence compared to vanilla SGD
- **CNN architecture**: Combines convolution, pooling, and dense layers for visual tasks

## Extension ideas

- Add batch normalization for faster training
- Try different architectures (deeper, wider, or with residual connections)
- Implement data augmentation for better generalization
- Add dropout for regularization
- Experiment with different learning rates and schedules